Ceci est le code de la première approche que nous avons eu pour le Jalon 3 qui était l'extension naturelle de la modélisation du Jalon 2, nous l'avons finalement abandonnée car elle n'est pas optimale en mémoire et ne permettait pas d'atteindre la limite optimale

# Importations nécessaires

In [1]:
import numpy as np
from gurobipy import *
import pandas as pd
from collections import defaultdict

In [2]:
#Hyperparamètres
acompte = 0.5 #Part d'acompte intial
duree_optimiste = 0 # 0 si durée totale; 1 si différence
budget_annuel = 2e6

#Poids pour la fonction objectif
poids_2030 = 0.25
poids_2040 = 0.25
poids_2050 = 0.5

sobriete = 0.1

# Préparation des données
Récuperation des matrices de gains, de coût, des listes des bâtiments, de leurs types et des durées de travaux.

In [3]:
# --- 1. Lecture du fichier ---
df = pd.read_excel("dataset_efficacity_avec_duree.xlsx")

# ============================
# PARTIE A — CALIBRATED
# ============================

df_calibrated = (
    df[df["id_simulation"].astype(str).str.contains("calibrated", case=False)]
    [["building_name", "typo_dpe", "id_simulation", "conso_total_mwh_an"]]
)

# Comptage par catégorie
nb_buildings_by_type = (
    df_calibrated
    .groupby("typo_dpe")["building_name"]
    .count()
    .to_dict()
)

# Liste des calibrated par bâtiment
calibrated_list = (
    df_calibrated
    .groupby("building_name")["conso_total_mwh_an"]
    .apply(list)
    .to_dict()
)

# ============================
# PARTIE B — SANS CALIBRATED
# ============================
df_nc = df[
    ~df["id_simulation"].astype(str).str.contains("calibrated", case=False)
]

# --- 2. Groupement ---
group_gains    = df_nc.groupby("building_name")["gains_totaux_mwh_an"].apply(list)
group_cost     = df_nc.groupby("building_name")["cout_investissement_euros"].apply(list)
group_duration = df_nc.groupby("building_name")["temps_de_travaux"].apply(list)  # en mois

# --- 3. Taille maximale ---
max_len = max(
    group_gains.apply(len).max(),
    group_cost.apply(len).max()
)

# --- 4. Matrices à taille fixe ---
gains_matrix = np.array([
    vals + [np.inf] * (max_len - len(vals))
    for vals in group_gains
])

cost_matrix = np.array([
    vals + [np.inf] * (max_len - len(vals))
    for vals in group_cost
])

group_type = (
    df_nc
    .groupby("building_name")["typo_dpe"]
    .first()
)

# --- 5. Métadonnées utiles ---
building_names = group_gains.index.tolist()
building_types = group_type.loc[building_names].tolist()

grouped = df_nc.groupby("building_name", sort=False)["id_simulation"].apply(list)
renovation_names_per_building = grouped.to_dict()

# Durée par simulation (en mois)
duration_by_sim = {
    row["id_simulation"]: int(row["temps_de_travaux"])
    for _, row in df_nc.iterrows()
}

df_calibrated = df_calibrated[
    df_calibrated["building_name"].isin(building_names)
]

## Construction du graphe de transitions

Pour un bâtiment b, on construit l'espace des transitions de la rénovation r vers la rénovation r' ssi $r\subset r'$

Alors la transition $r \rightarrow r'$ aura pour gain: gain(r') - gain(r) et pour coût: coût(r')-coût(r)

On initialise aussi des transitions initiales $None \rightarrow r$

**Nouveauté :** chaque transition porte également la durée $d_e$ (en mois) de la simulation cible.

In [4]:
# ============================
# PARTIE C — TRANSITION GRAPH
# ============================

transition_graph = {}
# Colonnes travaux (G → N)
renov_cols = df.columns[6:14]

# --- Construction par bâtiment ---
for building, group in df_nc.groupby("building_name", sort=False):

    group = group.reset_index(drop=True)

    works = group[renov_cols].astype(bool).values
    costs = group["cout_investissement_euros"].values
    gains = group["gains_totaux_mwh_an"].values
    ids   = group["id_simulation"].tolist()
    durs  = group["temps_de_travaux"].astype(int).values  # durées en mois

    n = len(group)

    # ========================
    # MATRICE COMPATIBILITE
    # ========================
    compat = np.zeros((n, n), dtype=int)
    for i in range(n):
        for j in range(n):
            if np.all(~works[j] | works[i]):
                compat[i, j] = 1

    # ========================
    # GENERATION DES TRANSITIONS
    # ========================
    transitions = []

    # état initial (aucun travaux)
    for i in range(n):
        transitions.append({
            "from": "none",
            "to":   ids[i],
            "cost": costs[i],
            "gain": gains[i],
            "duration": int(durs[i])   # durée en mois
        })

    # transitions entre rénovations
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            # j ⊂ i
            if compat[i, j] == 1 and compat[j, i] == 0:
                transitions.append({
                    "from": ids[j],
                    "to":   ids[i],
                    "cost": costs[i] - costs[j],
                    "gain": gains[i] - gains[j],
                    "duration": int(durs[i]) - duree_optimiste*int(durs[j])  # durée de la rénovation cible
                })

    transition_graph[building] = transitions

edges = {b: list(range(len(transition_graph[b]))) for b in building_names}

cost       = {}
gain       = {}
from_state = {}
to_state   = {}
duration   = {}   # durée en mois de chaque arête

for b in building_names:
    for e, edge in enumerate(transition_graph[b]):
        cost[b, e]       = edge["cost"]
        gain[b, e]       = edge["gain"]
        from_state[b, e] = edge["from"]
        to_state[b, e]   = edge["to"]
        duration[b, e]   = edge["duration"]

# ============================
# Isolation des travaux spécifiques (utile pour l'excel)
works_by_sim = {}
for _, row in df_nc.iterrows():
    works_by_sim[row["id_simulation"]] = {
        col: bool(row[col])
        for col in renov_cols
    }

In [13]:
print(transition_graph["abri_bouliste"][0])

{'from': 'none', 'to': 'abri_bouliste_simulation_1', 'cost': np.float64(9352.936), 'gain': np.float64(2.896469999999999), 'duration': 1}


In [14]:
# --- Exemple ---
example_building = list(transition_graph.keys())[3]
print("Building:", example_building)
for t in transition_graph[example_building][10:15]:
    print(t)

Building: gs_celestin_freinet
{'from': 'gs_celestin_freinet_simulation 3', 'to': 'gs_celestin_freinet_simulation 10', 'cost': np.float64(1163612.3625000003), 'gain': np.float64(259.41449833126325), 'duration': 8}
{'from': 'gs_celestin_freinet_simulation 4', 'to': 'gs_celestin_freinet_simulation 10', 'cost': np.float64(1272974.9125), 'gain': np.float64(246.89548647530353), 'duration': 11}
{'from': 'gs_celestin_freinet_simulation 5', 'to': 'gs_celestin_freinet_simulation 10', 'cost': np.float64(1010794.6625000001), 'gain': np.float64(234.32491523031098), 'duration': 0}
{'from': 'gs_celestin_freinet_simulation 6', 'to': 'gs_celestin_freinet_simulation 10', 'cost': np.float64(906862.55), 'gain': np.float64(94.55465173659408), 'duration': 0}
{'from': 'gs_celestin_freinet_simulation 7', 'to': 'gs_celestin_freinet_simulation 10', 'cost': np.float64(1059680.2500000002), 'gain': np.float64(118.01091485349477), 'duration': 4}


In [5]:
Conso_total  = np.sum(df_calibrated["conso_total_mwh_an"])
nbr_mois     = (2050 - 2026 + 1) * 12  # horizon en mois (t=0 → jan 2026, t=299 → déc 2050)
nbr_annees   = 2050 - 2026 + 1
nbr_batiments = gains_matrix.shape[0]

print("Conso total (avec sobriété 10%):", Conso_total, "MWh/an")
print("Nombre de bâtiments:", nbr_batiments)
print("Horizon:", nbr_mois, "mois")

Conso total (avec sobriété 10%): 20843.47743464322 MWh/an
Nombre de bâtiments: 71
Horizon: 300 mois


## Formulation alternative sans variables d'état

La granularité temporelle est le **mois** ($t=0$ = janvier 2026, $t=299$ = décembre 2050).

La seule variable de décision est :
$$y_{b,e,t} = 1 \quad \text{si la transition } e \text{ est démarrée sur le bâtiment } b \text{ au mois } t, \; 0 \text{ sinon}$$

Les variables $x_{b,t}$ et $z_{b,t}$ sont supprimées.

In [6]:
m = Model("Efficacity Optimization - Formulation Sans Variables Etat")
y = {}

for b in building_names:
    for e in edges[b]:
        d_e = duration[b, e]
        # Contrainte d'horizon : la rénovation doit se terminer avant t=nbr_mois
        t_max = nbr_mois - d_e
        for t in range(max(0, t_max + 1)):
            y[b, e, t] = m.addVar(
                vtype=GRB.BINARY,
                name=f"y_{b}_{e}_{t}"
            )

m.update()
print(f"Nombre de variables binaires : {m.NumVars}")

Set parameter Username
Set parameter LicenseID to value 2773884
Academic license - for non-commercial use only - expires 2027-02-02
Nombre de variables binaires : 745234


### Contrainte de non-chevauchement (sans variables d'état)

Pour tout bâtiment $b$ et tout mois $t$, au plus une rénovation peut être active :
$$\sum_e \sum_{\tau=t-d_e+1}^{t} y_{b,e,\tau} \leq 1$$

In [7]:
for b in building_names:
    for t in range(nbr_mois):
        terms = []
        for e in edges[b]:
            d_e = duration[b, e]
            # τ ∈ [t - d_e + 1, t] tels que y[b, e, τ] existe
            for tau in range(max(0, t - d_e + 1), t + 1):
                if (b, e, tau) in y:
                    terms.append(y[b, e, tau])
        if terms:
            m.addConstr(
                quicksum(terms) <= 1,
                name=f"no_overlap_{b}_{t}"
            )

### Contrainte d'unicité d'une rénovation
$$\forall b \quad \forall r \quad \sum_t \sum_{e,to(e)=r}y_{b,e,t} \leq 1$$

In [8]:
for b in building_names:
    states = set(edge["to"] for edge in transition_graph[b])
    for r in states:
        incoming = [
            e for e, edge in enumerate(transition_graph[b])
            if edge["to"] == r
        ]
        m.addConstr(
            quicksum(
                y[b, e, t]
                for e in incoming
                for t in range(nbr_mois)
                if (b, e, t) in y
            ) <= 1,
            name=f"state_once_{b}_{r}"
        )

### Contrainte : Quitter None une seule fois
$$\sum_t\sum_{e,from(e)=None}y_{b,e,t} \leq 1$$

In [9]:
for b in building_names:
    none_edges = [
        e for e, edge in enumerate(transition_graph[b])
        if edge["from"] == "none"
    ]
    m.addConstr(
        quicksum(
            y[b, e, t]
            for e in none_edges
            for t in range(nbr_mois)
            if (b, e, t) in y
        ) <= 1,
        name=f"leave_none_once_{b}"
    )

### Contrainte de budget annuel (avec répartition démarrage / fin)

Une fraction $acompte$ du coût est payée au démarrage, $1-acompte$ à la fin (mois $t + d_e$) :
$$\sum_{b,e,t : year(t)=a} acompte \cdot cost_e \cdot y_{b,e,t}
+ \sum_{b,e,t : year(t+d_e)=a} (1-acompte) \cdot cost_e \cdot y_{b,e,t} \leq Budget_a$$

In [10]:
def month_to_year(t):
    """Retourne l'index d'année (0 = 2026) correspondant au mois t."""
    return t // 12

for a in range(nbr_annees):
    # Mois appartenant à l'année a
    months_a = list(range(a * 12, (a + 1) * 12))

    m.addConstr(
        quicksum(
            acompte * cost[b, e] * y[b, e, t]
            for b in building_names
            for e in edges[b]
            for t in months_a
            if (b, e, t) in y
        )
        +
        quicksum(
            (1 - acompte) * cost[b, e] * y[b, e, t]
            for b in building_names
            for e in edges[b]
            for t in range(nbr_mois)
            if (b, e, t) in y
            and month_to_year(t + duration[b, e]) == a
        )
        <= budget_annuel,
        name=f"budget_annee_{a}"
    )

### Contrainte de disponibilité par catégorie

Pour tout mois $t$, au plus 20 % des bâtiments d'une catégorie peuvent être **en rénovation** :
$$\forall cat \quad \forall t \quad \sum_{b \in cat} \sum_e \sum_{\tau=t-d_e+1}^{t} y_{b,e,\tau} \leq 0.2 \cdot |cat|$$

**Exception :** en **janvier** (mois 0 mod 12), **juillet** (mois 6 mod 12) et **août** (mois 7 mod 12), la contrainte ne s'applique **pas** aux bâtiments de type **"Enseignement"**.

In [11]:
buildings_by_cat = defaultdict(list)
for b_name, cat in zip(building_names, building_types):
    buildings_by_cat[cat].append(b_name)

# Mois exemptés pour la catégorie Enseignement (janvier=0, juillet=6, août=7 dans l'année)
EXEMPT_MONTHS_OF_YEAR = {0, 6, 7}  # indices dans l'année (0-based)
EXEMPT_CATEGORY = "Enseignement"

for cat, b_names in buildings_by_cat.items():
    if cat == "Autres":
        continue

    card_cat = len(b_names)

    for t in range(nbr_mois):
        # Vérifier si ce mois est exempté pour la catégorie Enseignement
        month_in_year = t % 12
        is_exempt_month = month_in_year in EXEMPT_MONTHS_OF_YEAR

        if cat == EXEMPT_CATEGORY and is_exempt_month:
            # Pas de contrainte de disponibilité pour Enseignement en jan/jul/août
            continue

        # Nombre de bâtiments en rénovation au mois t
        # = sum sur b, e des y[b,e,tau] avec tau in [t-d_e+1, t]
        terms = []
        for b in b_names:
            for e in edges[b]:
                d_e = duration[b, e]
                for tau in range(max(0, t - d_e + 1), t + 1):
                    if (b, e, tau) in y:
                        terms.append(y[b, e, tau])

        if terms:
            m.addConstr(
                quicksum(terms) <= 0.2 * card_cat,
                name=f"availability_{cat}_{t}"
            )

### Contrainte de cohérence des flux (formulation compacte sans variables auxiliaires)

**Contexte :** la contrainte de flux
$$\sum_{\tau \leq t}\sum_{e:to(e)=r}y_{b,e,\tau} \geq \sum_{\tau \leq t}\sum_{e:from(e)=r}y_{b,e,\tau}$$

> *On ne peut quitter l'état $r$ que si on y est arrivé.*

In [12]:
for b in building_names:

    states = set()

    for edge in transition_graph[b]:
        states.add(edge["from"])
        states.add(edge["to"])

    states.discard("none")

    for r in states:

        incoming = [
            e for e, edge in enumerate(transition_graph[b])
            if edge["to"] == r
        ]

        outgoing = [
            e for e, edge in enumerate(transition_graph[b])
            if edge["from"] == r
        ]

        for t in range(nbr_mois):

            m.addConstr(
                quicksum(
                    y[b,e,tau] for e in incoming for tau in range(t+1) if (b, e, tau) in y
                )
                >=
                quicksum(
                    y[b,e,tau] for e in outgoing for tau in range(t+1) if (b, e, tau) in y
                ),
                name=f"flow_{b}_{r}_{t}"
            )

GurobiError: Out of memory

### Fonction Objectif

Les gains sont pris en compte à la **fin** de la rénovation, soit au mois $t + d_e$ :

$$\min \left[
0.25\left(0.4 C_0 - \sum_{b,e,t : t+d_e \leq T_{2030}} gain_{b,e} \cdot y_{b,e,t}\right)
+ 0.25\left(0.5 C_0 - \sum_{b,e,t : t+d_e \leq T_{2040}} gain_{b,e} \cdot y_{b,e,t}\right)
+ 0.5\left(0.6 C_0 - \sum_{b,e,t : t+d_e \leq T_{2050}} gain_{b,e} \cdot y_{b,e,t}\right)
\right]$$

où $T_{2030}$, $T_{2040}$, $T_{2050}$ désignent le dernier mois de 2030, 2040 et 2050 respectivement.

In [23]:
# Dernier mois inclus dans chaque période (fin décembre)
T_2030 = (2030 - 2026 + 1) * 12 - 1   # = 59  (décembre 2030)
T_2040 = (2040 - 2026 + 1) * 12 - 1   # = 179 (décembre 2040)
T_2050 = nbr_mois - 1                  # = 299 (décembre 2050)

def gain_before(T):
    """Somme des gains dont la fin de rénovation (t + d_e) est <= T."""
    return quicksum(
        gain[b, e] * y[b, e, t]
        for b in building_names
        for e in edges[b]
        for t in range(nbr_mois)
        if (b, e, t) in y and (t + duration[b, e] - 1) <= T
    )

m.setObjective(
      poids_2030*(gain_before(T_2030))
    + poids_2040*(gain_before(T_2040))
    + poids_2050*(gain_before(T_2050)),
    GRB.MAXIMIZE
)

In [24]:
# Fonction objectif alternative : maximiser les gains en 2050
""" m.setObjective(gain_before(T_2050), GRB.MAXIMIZE) """

' m.setObjective(gain_before(T_2050), GRB.MAXIMIZE) '

In [ ]:
m.update()
print("Nombre de variables du modèle: ", m.NumVars)
print("Nombre de contraintes du modèle: ", m.NumConstrs)
print(f"Nombre de coefficients non nuls dans la matrice de contraintes: {m.NumNZs:,}")

Nombre de variables du modèle:  749803
Nombre de contraintes du modèle:  257721


### Résolution du modèle

In [ ]:
# Remarque: Time Limit nécessaire sinon le modèle ne termine pas
# Au-delà de 2 min le gain marginal est faible
m.setParam('TimeLimit', 2 * 60)
m.update()
m.optimize()

### Valeur de la fonction objectif

In [ ]:
print(f"Valeur objectif : {m.ObjVal:.4f}")

Valeur objectif : 12832.1204


### Objectifs réalisés (avec sobriété 10%)

Les gains sont comptabilisés **à la fin** des rénovations, cohérent avec la formulation sans variables d'état.

In [ ]:
def realized_gain_before(T):
    return sum(
        y[b, e, t].X * gain[b, e]
        for b in building_names
        for e in edges[b]
        for t in range(nbr_mois)
        if (b, e, t) in y and (t + duration[b, e] - 1) <= T
    )

obj_2030 = sobriete + (1 - sobriete) * realized_gain_before(T_2030) / Conso_total
obj_2040 = sobriete + (1 - sobriete) * realized_gain_before(T_2040) / Conso_total
obj_2050 = sobriete + (1 - sobriete) * realized_gain_before(T_2050) / Conso_total

print(f"Objectif 2030 : {obj_2030:.4f} (cible 0.4)")
print(f"Objectif 2040 : {obj_2040:.4f} (cible 0.5)")
print(f"Objectif 2050 : {obj_2050:.4f} (cible 0.6)")

Objectif 2030 : 0.2279 (cible 0.4)
Objectif 2040 : 0.3910 (cible 0.5)
Objectif 2050 : 0.4771 (cible 0.6)


In [ ]:
def works_added(from_state, to_state):
    """Isolation des travaux spécifiques à partir d'une arête du graphe.
    Utile pour connaître exactement quelles rénovations sont faites."""
    if from_state == "none":
        return [
            col for col, v in works_by_sim[to_state].items() if v
        ]

    return [
        col
        for col in renov_cols
        if works_by_sim[to_state][col] and not works_by_sim[from_state][col]
    ]


# =========================
# Paramètres temporels
# =========================

start_year = 2026
end_year = 2050
nbr_years = end_year - start_year + 1
nbr_months = 12 * nbr_years

month_labels = [
    f"{start_year + t // 12}-{(t % 12) + 1:02d}"
    for t in range(nbr_months)
]

year_labels = list(range(start_year, end_year + 1))


def month_to_year_month_period(t):
    """Convertit un indice mensuel t en (year, month, period)."""
    year = start_year + t // 12
    month = (t % 12) + 1
    period = f"{year}-{month:02d}"
    return year, month, period


def month_to_year(t):
    """Convertit un indice mensuel t en indice d'année relatif."""
    return t // 12


# =========================
# Construction des tables
# =========================
# df_starts : seulement les mois de démarrage
# df_active : tous les mois où y[b,e,t] = 1
# =========================

rows_starts = []
rows_active = []

for (b, e, t), var in y.items():
    if var.X > 0.5:
        tr = transition_graph[b][e]
        added = works_added(tr["from"], tr["to"])
        year, month, period = month_to_year_month_period(t)

        row_common = {
            "building": b,
            "transition": e,
            "month_index": t,
            "year": year,
            "month": month,
            "period": period,
            "from_state": tr["from"],
            "to_state": tr["to"],
            "transition_label": f'{tr["from"]} -> {tr["to"]}',
            "works_added": ", ".join(added),
            "gain": tr["gain"],
            "cost": tr["cost"]
        }

        rows_active.append(row_common.copy())

        prev_active = 0
        if t > 0 and (b, e, t - 1) in y:
            prev_active = 1 if y[b, e, t - 1].X > 0.5 else 0

        if prev_active == 0:
            rows_starts.append(row_common.copy())

df_starts = pd.DataFrame(rows_starts)
df_active = pd.DataFrame(rows_active)

if df_starts.empty:
    df_starts = pd.DataFrame(columns=[
        "building", "transition", "month_index", "year", "month", "period",
        "from_state", "to_state", "transition_label", "works_added", "gain", "cost"
    ])

if df_active.empty:
    df_active = pd.DataFrame(columns=[
        "building", "transition", "month_index", "year", "month", "period",
        "from_state", "to_state", "transition_label", "works_added", "gain", "cost"
    ])

df_starts = df_starts.sort_values(["building", "month_index"]).reset_index(drop=True)
df_active = df_active.sort_values(["building", "month_index"]).reset_index(drop=True)


# =========================
# Timeline des transitions démarrées
# =========================

timeline = (
    df_starts
    .pivot_table(
        index="building",
        columns="period",
        values="transition_label",
        aggfunc="first"
    )
    .reindex(columns=month_labels)
)

# optionnel : uniquement les états d'arrivée
timeline_to_state = (
    df_starts
    .pivot_table(
        index="building",
        columns="period",
        values="to_state",
        aggfunc="first"
    )
    .reindex(columns=month_labels)
)

# travaux ajoutés au démarrage
timeline_works = (
    df_starts
    .groupby(["building", "period"])["works_added"]
    .agg(lambda x: " | ".join(sorted(set(v for v in x if pd.notna(v) and v != ""))))
    .unstack()
    .reindex(columns=month_labels)
)


# =========================
# Reconstruction de l'état courant
# mois par mois
# =========================

state_rows = []

for b in building_names:
    sub = (
        df_starts[df_starts["building"] == b]
        .sort_values("month_index")
        .reset_index(drop=True)
    )

    starts_by_month = {
        row["month_index"]: row for _, row in sub.iterrows()
    }

    current_state = "none"

    for t in range(nbr_months):
        year, month, period = month_to_year_month_period(t)

        transition_started = ""
        works_started = ""
        is_coherent = True

        if t in starts_by_month:
            row = starts_by_month[t]
            transition_started = row["transition_label"]
            works_started = row["works_added"]
            is_coherent = (row["from_state"] == current_state)

            if is_coherent:
                current_state = row["to_state"]
            else:
                current_state = f'INCOHERENT({row["from_state"]} -> {row["to_state"]})'

        state_rows.append({
            "building": b,
            "month_index": t,
            "year": year,
            "month": month,
            "period": period,
            "current_state": current_state,
            "transition_started": transition_started,
            "works_started": works_started,
            "is_coherent_transition": is_coherent
        })

df_state_timeline = pd.DataFrame(state_rows)

timeline_states_rebuilt = (
    df_state_timeline
    .pivot_table(
        index="building",
        columns="period",
        values="current_state",
        aggfunc="first"
    )
    .reindex(columns=month_labels)
)


# =========================
# Stats par mois
# basées sur les démarrages
# =========================

all_months_index = pd.MultiIndex.from_tuples(
    [
        (
            start_year + t // 12,
            (t % 12) + 1,
            f"{start_year + t // 12}-{(t % 12) + 1:02d}"
        )
        for t in range(nbr_months)
    ],
    names=["year", "month", "period"]
)

stats_months = (
    df_starts
    .groupby(["year", "month", "period"])
    .agg(
        renovations=("building", "count"),
        gain_total=("gain", "sum"),
        cost_total_started=("cost", "sum")
    )
    .reindex(all_months_index, fill_value=0)
    .reset_index()
)


# =========================
# Stats par année
# budget réellement payé dans l'année
# =========================

stats_years_base = (
    df_starts
    .groupby("year")
    .agg(
        renovations=("building", "count"),
        gain_total=("gain", "sum")
    )
    .reindex(year_labels, fill_value=0)
    .reset_index()
)

budget_rows = []

for a in range(nbr_years):
    year = start_year + a
    months_a = list(range(a * 12, (a + 1) * 12))

    budget_start_paid = sum(
        acompte * cost[b, e]
        for b in building_names
        for e in edges[b]
        for t in months_a
        if (b, e, t) in y and y[b, e, t].X > 0.5
    )

    budget_end_paid = sum(
        (1 - acompte) * cost[b, e]
        for b in building_names
        for e in edges[b]
        for t in range(nbr_months)
        if (
            (b, e, t) in y
            and y[b, e, t].X > 0.5
            and month_to_year(t + duration[b, e]) == a
        )
    )

    budget_rows.append({
        "year": year,
        "budget_start_paid": budget_start_paid,
        "budget_end_paid": budget_end_paid,
        "cost_total": budget_start_paid + budget_end_paid
    })

budget_df = pd.DataFrame(budget_rows)

stats_years = stats_years_base.merge(budget_df, on="year", how="left")
stats_years["budget_start_paid"] = stats_years["budget_start_paid"].fillna(0)
stats_years["budget_end_paid"] = stats_years["budget_end_paid"].fillna(0)
stats_years["cost_total"] = stats_years["cost_total"].fillna(0)
stats_years["budget_limit"] = budget_annuel
stats_years["budget_respected"] = stats_years["cost_total"] <= budget_annuel + 1e-6


# =========================
# Vérification de cohérence
# des enchaînements de transitions
# =========================

coherence_rows = []

for b in df_starts["building"].unique():
    sub = (
        df_starts[df_starts["building"] == b]
        .sort_values("month_index")
        .reset_index(drop=True)
    )

    current_state = "none"

    for _, row in sub.iterrows():
        is_coherent = (current_state == row["from_state"])

        coherence_rows.append({
            "building": b,
            "period": row["period"],
            "month_index": row["month_index"],
            "expected_from_state": current_state,
            "actual_from_state": row["from_state"],
            "to_state": row["to_state"],
            "transition_label": row["transition_label"],
            "is_coherent": is_coherent
        })

        current_state = row["to_state"]

df_coherence = pd.DataFrame(coherence_rows)

if df_coherence.empty:
    df_coherence = pd.DataFrame(columns=[
        "building", "period", "month_index",
        "expected_from_state", "actual_from_state",
        "to_state", "transition_label", "is_coherent"
    ])


# =========================
# Disponibilité par catégorie et par mois
# basée sur les mois actifs
# =========================

building_type_map = dict(zip(building_names, building_types))
df_active["category"] = df_active["building"].map(building_type_map)

categories = list(nb_buildings_by_type.keys())

all_category_periods = pd.MultiIndex.from_product(
    [categories, month_labels],
    names=["category", "period"]
)

availability = (
    df_active
    .groupby(["category", "period"])
    .size()
    .reindex(all_category_periods, fill_value=0)
    .reset_index(name="renovations_done")
)

availability["year"] = availability["period"].str[:4].astype(int)
availability["month"] = availability["period"].str[5:7].astype(int)

availability["max_allowed"] = availability["category"].map(
    lambda c: 0.2 * nb_buildings_by_type[c]
)

availability["usage_ratio"] = availability["renovations_done"] / availability["max_allowed"]
availability.loc[availability["max_allowed"] == 0, "usage_ratio"] = 0


# =========================
# Timeline activité mensuelle
# utile pour voir quels bâtiments sont en chantier chaque mois
# =========================

timeline_active = (
    df_active
    .groupby(["building", "period"])["to_state"]
    .agg(lambda x: " | ".join(sorted(set(v for v in x if pd.notna(v)))))
    .unstack()
    .reindex(columns=month_labels)
)

timeline_active_works = (
    df_active
    .groupby(["building", "period"])["works_added"]
    .agg(lambda x: " | ".join(sorted(set(v for v in x if pd.notna(v) and v != ""))))
    .unstack()
    .reindex(columns=month_labels)
)


# =========================
# Export Excel
# =========================

with pd.ExcelWriter("plan_renovation_solution.xlsx") as writer:
    df_starts.to_excel(writer, sheet_name="transitions", index=False)
    df_active.to_excel(writer, sheet_name="active_months", index=False)

    timeline.to_excel(writer, sheet_name="timeline_renovations")
    timeline_to_state.to_excel(writer, sheet_name="timeline_to_state")
    timeline_works.to_excel(writer, sheet_name="timeline_travaux")
    timeline_states_rebuilt.to_excel(writer, sheet_name="timeline_states_rebuilt")

    timeline_active.to_excel(writer, sheet_name="timeline_active_states")
    timeline_active_works.to_excel(writer, sheet_name="timeline_active_works")

    stats_months.to_excel(writer, sheet_name="stats_mois", index=False)
    stats_years.to_excel(writer, sheet_name="stats_annees", index=False)

    df_coherence.to_excel(writer, sheet_name="coherence_checks", index=False)
    df_state_timeline.to_excel(writer, sheet_name="state_timeline_detail", index=False)
    availability.to_excel(writer, sheet_name="availability_by_category", index=False)